In [1]:
!pip install -q matplotlib seaborn scikit-learn

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/food_dataset_complete"

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VALIDATE_DIR = os.path.join(DATASET_DIR, 'val')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

print("=======Dataset Summary=======")

for split in ['train', 'val', 'test']:
    print("\n", split)

    path = os.path.join(DATASET_DIR, split)

    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)


        if not os.path.isdir(cls_path):
            continue

        count = len(os.listdir(cls_path))
        print(cls, ":", count, "photo")

=======Dataset Summary=======

 train
thaisourcurry : 103 photo
tomyum : 104 photo
padthai : 167 photo
somtam : 188 photo
panangchickencurry : 105 photo
larb : 102 photo
tomkhagai : 131 photo
padkrapao : 180 photo
massamancurry : 103 photo
greencurry : 176 photo

 val
panangchickencurry : 49 photo
larb : 47 photo
padkrapao : 62 photo
tomkhagai : 59 photo
tomyum : 49 photo
padthai : 61 photo
thaisourcurry : 55 photo
massamancurry : 46 photo
greencurry : 69 photo
somtam : 68 photo

 test
massamancurry : 19 photo
greencurry : 32 photo
padthai : 28 photo
larb : 20 photo
thaisourcurry : 19 photo
panangchickencurry : 20 photo
padkrapao : 32 photo
tomyum : 20 photo
tomkhagai : 23 photo
somtam : 32 photo


In [4]:
IMG_SIZE = 224           # EfficientNetB0 ใช้ 224x224
BATCH_SIZE  = 16         # แบ่งรูปเทรนต่อรอบ
EPOCHS_TOP  = 15
EPOCHS_FINE = 15         # Fine-tune ทั้ง model
LR_TOP      = 5e-4       # Learning rate ช่วงแรก
LR_FINE     = 1e-4       # Learning rate ช่วง fine-tune
OUTPUT_DIR  = '/content/benchmark_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAME = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith('.')])
NUM_CLASS = len(CLASS_NAME)

print("=======Config=======")
print(f' Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f' Batch size : {BATCH_SIZE}')
print(f' Epochs top : {EPOCHS_TOP}')
print(f' Epochs fine: {EPOCHS_FINE}')
print(f' Learning rate top : {LR_TOP}')
print(f' Learning rate fine: {LR_FINE}')
print(f' Amout Class  : {NUM_CLASS}')
print(f' Class name : {CLASS_NAME}')

=======Config=======
 Image size  : 224x224
 Batch size : 16
 Epochs top : 15
 Epochs fine: 15
 Learning rate top : 0.0005
 Learning rate fine: 0.0001
 Amout Class  : 10
 Class name : ['greencurry', 'larb', 'massamancurry', 'padkrapao', 'padthai', 'panangchickencurry', 'somtam', 'thaisourcurry', 'tomkhagai', 'tomyum']


In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
import os

TEST_DIR = os.path.join(DATASET_DIR, 'test')

train_data = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    brightness_range = [0.8, 1.2],
    fill_mode = 'nearest'
)

val_data = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

test_data = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

# load
train_loader = train_data.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle=True
)

val_loader = val_data.flow_from_directory(
    VALIDATE_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)

test_loader = test_data.flow_from_directory(
    TEST_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)

print('Class indices:', train_loader.class_indices)

Found 1358 images belonging to 10 classes.
Found 565 images belonging to 10 classes.
Found 245 images belonging to 10 classes.
Class indices: {'greencurry': 0, 'larb': 1, 'massamancurry': 2, 'padkrapao': 3, 'padthai': 4, 'panangchickencurry': 5, 'somtam': 6, 'thaisourcurry': 7, 'tomkhagai': 8, 'tomyum': 9}


In [6]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications import InceptionV3

from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2  import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mob_pre
from tensorflow.keras.applications.densenet import preprocess_input as dense_pre
from tensorflow.keras.applications.inception_v3 import preprocess_input as inc_pre

MODELS_CONFIG = [
    {
        'name':         'EfficientNetB0',
        'base_class':   EfficientNetB0,
        'preprocess':   eff_pre,
        'img_size':     224,
    },

    {
        'name':         'ResNet50V2',
        'base_class':   ResNet50V2,
        'preprocess':   res_pre,
        'img_size':     224,
    },

    {
        'name':         'MobileNetV3Large',
        'base_class':   MobileNetV3Large,
        'preprocess':   mob_pre,
        'img_size':     224,
    },

    {
        'name':         'DenseNet121',
        'base_class':   DenseNet121,
        'preprocess':   dense_pre,
        'img_size':     224,
    },

    {
        'name':         'InceptionV3',
        'base_class':   InceptionV3,
        'preprocess':   inc_pre,
        'img_size':     299,
    },
]
print(f'Number of models to training: {len(MODELS_CONFIG)} ')

Number of models to training: 5 


In [7]:
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def make_loaders(preprocess_function, img_size):
    train_data = ImageDataGenerator(
        preprocessing_function = preprocess_function,
        rotation_range = 20,
        width_shift_range = 0.2,
        height_shift_range = 0.2,
        shear_range = 0.2,
        zoom_range = 0.2,
        horizontal_flip = True,
        brightness_range = [0.8, 1.2],
        fill_mode = 'nearest'
    )

    val_data = ImageDataGenerator(
        preprocessing_function = preprocess_function
    )

    test_data = ImageDataGenerator(
        preprocessing_function = preprocess_function
    )

    train_loader = train_data.flow_from_directory(
        TRAIN_DIR,
        target_size=(img_size, img_size),
        batch_size = BATCH_SIZE,
        class_mode = 'categorical',
        shuffle = True
    )

    val_loader = val_data.flow_from_directory(
        VALIDATE_DIR,
        target_size=(img_size, img_size),
        batch_size = BATCH_SIZE,
        class_mode = 'categorical',
        shuffle = False
    )

    test_loader = test_data.flow_from_directory(
        TEST_DIR,
        target_size=(img_size, img_size),
        batch_size = BATCH_SIZE,
        class_mode = 'categorical',
        shuffle = False
    )
    return train_loader, val_loader, test_loader

def create_model(base_model_class, img_size, num_classes):
    base_model = base_model_class(
        weights ='imagenet',
        include_top = False,
        input_shape = (img_size, img_size, 3)
    )

    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation ='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(num_classes, activation ='softmax')(x)

    model = models.Model(
        inputs = base_model.input,
        outputs = output

    )

    return model, base_model


# callbacks save model
def get_callbacks(model_name):
    save_folder = "models"
    os.makedirs(save_folder, exist_ok = True)

    save_path = os.path.join(save_folder, model_name + ".keras") # Changed .h5 to .keras

    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath = save_path,
            monitor = 'val_accuracy',
            save_best_only = True
        ),
        tf.keras.callbacks.EarlyStopping(
            patience = 3,
            restore_best_weights = True
        )
    ]


# train
def train_model(config):
    model_name = config['name']
    img_size = config['img_size']

    print("Train model:", model_name)

    # โหลด data
    train_data, val_data, test_data = make_loaders(config['preprocess'], img_size)

    # สร้าง model
    model, base_model = create_model(config['base_model'], img_size, NUM_CLASS)

    # Phase 1
    print("=====Phase 1 (train head)=====")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_TOP),
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    history1 = model.fit(
        train_data,
        validation_data = val_data,
        epochs = EPOCHS_TOP,
        callbacks = get_callbacks(model_name + "_phase1")
    )

    # Phase 2
    print("=====Phase 2 (fine-tune)=====")

    base_model.trainable = True

    model.compile(
        optimizer = tf.keras.optimizers.Adam(LR_FINE),
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    history2 = model.fit(
        train_data,
        validation_data=val_data,
        epochs = EPOCHS_TOP,
        callbacks = get_callbacks(model_name)
    )

    return model, test_data, history1, history2

In [8]:
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

def evaluate_model(model, test_data):
    print("=====Evaluate model=====")

    model_outputs = []
    correct_answers = []

    for i in range(len(test_data)):
        x, y = test_data[i]
        preds = model.predict(x)
        model_outputs.append(preds)
        correct_answers.append(np.argmax(y, axis=1))


    model_outputs = np.vstack(model_outputs)
    correct_answers = np.concatenate(correct_answers)


    predicted_class = np.argmax(model_outputs, axis=1)

    # Accuracy
    accuracy = np.mean(predicted_class == correct_answers) * 100

    # F1-score
    f1 = f1_score(correct_answers, predicted_class, average='macro') * 100

    # confusion_matrix
    cm = confusion_matrix(correct_answers, predicted_class)

    print("Accuracy:", round(accuracy, 2), "%")
    print("F1-score:", round(f1, 2), "%")
    print("Confusion Matrix:", cm)

    return accuracy, f1, cm

In [9]:
import os
import gc
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.inception_v3 import preprocess_input as inc_pre
from tensorflow.keras.applications.densenet import preprocess_input as dense_pre

save_folder = "models"
os.makedirs(save_folder, exist_ok=True)

model_list = [
    ("EfficientNetB0", tf.keras.applications.EfficientNetB0, 224, eff_pre),
    ("MobileNetV2",    tf.keras.applications.MobileNetV2,    224, mob_pre),
    ("ResNet50",       tf.keras.applications.ResNet50,        224, res_pre),
    ("DenseNet121",    tf.keras.applications.DenseNet121,     224, dense_pre),
    ("InceptionV3",    tf.keras.applications.InceptionV3,     299, inc_pre),
]

results = []

for name, base_model, img_size, preprocess in model_list:
    config = {
        'name': name,
        'img_size': img_size,
        'base_model': base_model,
        'preprocess': preprocess,
    }

    model, test_data, h1, h2 = train_model(config)

    acc, f1, cm = evaluate_model(model, test_data)
    print(name, "→", acc, f1)

    results.append((name, acc, f1))

    model_path = os.path.join(save_folder, name + ".keras")
    model.save(model_path)
    print("Saved model →", model_path)

    # ล้าง memory หลังแต่ละ model
    del model, test_data, h1, h2, cm
    tf.keras.backend.clear_session()
    gc.collect()
    print(f"Cleared memory after {name}\n")

df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1"])
df = df.sort_values(by="Accuracy", ascending=False)
df.to_csv("results.csv", index=False)

print("\n===== FINAL RESULT =====", df)

Train model: EfficientNetB0
Found 1358 images belonging to 10 classes.
Found 565 images belonging to 10 classes.
Found 245 images belonging to 10 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
=====Phase 1 (train head)=====
Epoch 1/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 1352s 16s/step - accuracy: 0.4838 - loss: 1.5792 - val_accuracy: 0.8000 - val_loss: 0.8301
Epoch 2/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 42s 491ms/step - accuracy: 0.7437 - loss: 0.8230 - val_accuracy: 0.8319 - val_loss: 0.5672
Epoch 3/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 41s 487ms/step - accuracy: 0.8027 - loss: 0.6104 - val_accuracy: 0.9062 - val_loss: 0.3870
Epoch 4/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 41s 477ms/step - accuracy: 0.8233 - loss: 0.5125 - val_accuracy: 0.9062 - val_loss: 0.3550
Epoch 5/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 41s 489ms/step - accuracy: 0.8697 - loss: 0.4269 - val_accuracy: 0.9168 - val_loss: 0.3075
Epoch 6/15
85/85 ━━━━━━━━━━━━━━━━━━━━ 82s 489ms/step - accuracy: 0.8763 - loss: 0.3870 - val_accuracy: 0.9221 - val_lo

In [10]:
import shutil
from google.colab import files

shutil.make_archive("all_files", 'zip', save_folder)
files.download("all_files.zip")

print('Download file complete')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download file complete
